<a href="https://colab.research.google.com/github/kiransree2002/NLP_FakeNews-Classification/blob/main/Model/FakeNews(Classification)_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Mounting Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Libraries

In [4]:
import pandas as pd
import numpy as np
import nltk
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Reading Dataset

In [6]:
train_df = pd.read_csv("/content/train (2).csv", sep=";")
#sep = separator
# ";" = semicolon delimiter
train_df.head()

,Unnamed: 0,title,text,label
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [7]:
eval_df = pd.read_csv("/content/evaluation.csv", sep=";")
train_df.head()

,Unnamed: 0,title,text,label
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [8]:
test_df = pd.read_csv("/content/test (1).csv", sep=";")
train_df.head()

,Unnamed: 0,title,text,label
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [9]:
# By looking on the head() - result , we might think about all the datasets are either same or not . for avoiding that confusion we can check it
print(train_df.equals(eval_df))
print(train_df.equals(test_df))
print(eval_df.equals(test_df))
#if any of them prints "True" , then they might be same

False
False
False


# EDA

## EDA for Training dataset

In [10]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24353 entries, 0 to 24352
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  24353 non-null  int64 
 1   title       24353 non-null  object
 2   text        24353 non-null  object
 3   label       24353 non-null  int64 
dtypes: int64(2), object(2)
memory usage: 761.2+ KB


In [11]:
train_df.head()

,Unnamed: 0,title,text,label
0,0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [12]:
train_df.shape

(24353, 4)

## EDA for Evaluation dataset

In [13]:
eval_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8117 entries, 0 to 8116
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  8117 non-null   int64 
 1   title       8117 non-null   object
 2   text        8117 non-null   object
 3   label       8117 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 253.8+ KB


In [14]:
eval_df.head()

,Unnamed: 0,title,text,label
0,0,"Sanders back in U.S. Senate, blasts 'coloniali...",WASHINGTON (Reuters) - Democratic U.S. preside...,1
1,1,Kremlin: Syria peoples' congress being 'active...,MOSCOW (Reuters) - A proposal to convene a con...,1
2,2,Oregon Cop Convicted Of Shattering Biker’s Co...,"In a baffling fit of rage, an Oregon State Pol...",0
3,3,Twitter Erupts With Glee Over #CruzSexScandal...,The last thing any politician running for the ...,0
4,4,MUST WATCH VIDEO: Obama Tries To Trash Trump B...,This is too good to miss! Mr. Teleprompter did...,0


In [15]:
eval_df.shape

(8117, 4)

## EDA for Test dataset

In [16]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8117 entries, 0 to 8116
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Unnamed: 0  8117 non-null   int64 
 1   title       8117 non-null   object
 2   text        8117 non-null   object
 3   label       8117 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 253.8+ KB


In [17]:
test_df.head()

,Unnamed: 0,title,text,label
0,0,"Live from New York, it's a Trump-Clinton remat...",NEW YORK (Reuters) - Veteran actor and frequen...,1
1,1,Catalan separatists to lose majority in tight ...,BARCELONA (Reuters) - Catalonia s independence...,1
2,2,North Carolina governor concedes election to D...,"WINSTON-SALEM, N.C. (Reuters) - North Carolina...",1
3,3,Draft Senate Iran legislation sets tough new U...,WASHINGTON (Reuters) - Draft legislation respo...,1
4,4,California governor taps U.S. Representative B...,"SACRAMENTO, Calif. (Reuters) - California Gove...",1


In [18]:
test_df.shape

(8117, 4)

# Preprocessing

## Dropping the unwanted columns

In [19]:
train_df = train_df.drop(train_df.columns[0], axis=1)
eval_df = eval_df.drop(eval_df.columns[0], axis=1)
test_df = test_df.drop(test_df.columns[0], axis=1)

In [20]:
# Checking whether the unwanted first column is removed or not
train_df.head()

,title,text,label
0,Palestinians switch off Christmas lights in Be...,"RAMALLAH, West Bank (Reuters) - Palestinians s...",1
1,China says Trump call with Taiwan president wo...,BEIJING (Reuters) - U.S. President-elect Donal...,1
2,FAIL! The Trump Organization’s Credit Score W...,While the controversy over Trump s personal ta...,0
3,Zimbabwe military chief's China trip was norma...,BEIJING (Reuters) - A trip to Beijing last wee...,1
4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,There has never been a more UNCOURAGEOUS perso...,0


In [21]:
# Again verifying whether the unwanted first column is removed or not for eval_df
eval_df.head()

,title,text,label
0,"Sanders back in U.S. Senate, blasts 'coloniali...",WASHINGTON (Reuters) - Democratic U.S. preside...,1
1,Kremlin: Syria peoples' congress being 'active...,MOSCOW (Reuters) - A proposal to convene a con...,1
2,Oregon Cop Convicted Of Shattering Biker’s Co...,"In a baffling fit of rage, an Oregon State Pol...",0
3,Twitter Erupts With Glee Over #CruzSexScandal...,The last thing any politician running for the ...,0
4,MUST WATCH VIDEO: Obama Tries To Trash Trump B...,This is too good to miss! Mr. Teleprompter did...,0


In [22]:
# Again cross verifying whether the unwanted first column is removed or not for test_df
test_df.head()

,title,text,label
0,"Live from New York, it's a Trump-Clinton remat...",NEW YORK (Reuters) - Veteran actor and frequen...,1
1,Catalan separatists to lose majority in tight ...,BARCELONA (Reuters) - Catalonia s independence...,1
2,North Carolina governor concedes election to D...,"WINSTON-SALEM, N.C. (Reuters) - North Carolina...",1
3,Draft Senate Iran legislation sets tough new U...,WASHINGTON (Reuters) - Draft legislation respo...,1
4,California governor taps U.S. Representative B...,"SACRAMENTO, Calif. (Reuters) - California Gove...",1


##Combining('Title' + 'Text') - Instead of preprocessing them separately

In [23]:
# Instead of preprocessing title and text separately, combine them(Combine 'Title' + 'Text') for "train_df"

train_df["content"] = train_df["title"] + " " + train_df["text"]
train_df["content"]

,content
0,Palestinians switch off Christmas lights in Be...
1,China says Trump call with Taiwan president wo...
2,FAIL! The Trump Organization’s Credit Score W...
3,Zimbabwe military chief's China trip was norma...
4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...
...,...
24348,Mexico Senate committee OK's air transport dea...
24349,BREAKING: HILLARY CLINTON’S STATE DEPARTMENT G...
24350,trump breaks from stump speech to admire beaut...
24351,NFL PLAYER Delivers Courageous Message: Stop B...


In [24]:
# Instead of preprocessing title and text separately, combine them(Combine 'Title' + 'Text') for eval_df"

eval_df["content"] = eval_df["title"] + " " + eval_df["text"]
eval_df["content"]

,content
0,"Sanders back in U.S. Senate, blasts 'coloniali..."
1,Kremlin: Syria peoples' congress being 'active...
2,Oregon Cop Convicted Of Shattering Biker’s Co...
3,Twitter Erupts With Glee Over #CruzSexScandal...
4,MUST WATCH VIDEO: Obama Tries To Trash Trump B...
...,...
8112,Sean Hannity Throws Hissy Fit After Real Repo...
8113,FORMER ASST FBI DIRECTOR WARNS ANTI-TRUMP KABA...
8114,John McCain: Trump’s Attacks On The Press Are...
8115,Syria's Deir al-Zor air base working again: st...


In [25]:
# Instead of preprocessing title and text separately, combine them(Combine 'Title' + 'Text') for "testt_df"

test_df["content"] = test_df["title"] + " " + test_df["text"]
test_df["content"]

,content
0,"Live from New York, it's a Trump-Clinton remat..."
1,Catalan separatists to lose majority in tight ...
2,North Carolina governor concedes election to D...
3,Draft Senate Iran legislation sets tough new U...
4,California governor taps U.S. Representative B...
...,...
8112,Sanders at Vatican says rich-poor gap worse th...
8113,how trump happened force and fanaticism wahha...
8114,Turkey will take two steps if Germany takes on...
8115,BREAKING: DEVICE THAT BUSTED HILLARY CLINTON D...


##Lowercase

In [26]:
def lowercase(text):
    return text.lower()

In [27]:
train_df["lowercase"] = train_df["content"].apply(lowercase)
train_df["lowercase"]

,lowercase
0,palestinians switch off christmas lights in be...
1,china says trump call with taiwan president wo...
2,fail! the trump organization’s credit score w...
3,zimbabwe military chief's china trip was norma...
4,the most uncourageous president ever receives ...
...,...
24348,mexico senate committee ok's air transport dea...
24349,breaking: hillary clinton’s state department g...
24350,trump breaks from stump speech to admire beaut...
24351,nfl player delivers courageous message: stop b...


##Removing HTML Tags

In [28]:
def remove_html(text):
    clean = re.compile("<.*?>")
    return re.sub(clean, "", text)

In [29]:
train_df["html_removed"] = train_df["lowercase"].apply(remove_html)
train_df["html_removed"]

,html_removed
0,palestinians switch off christmas lights in be...
1,china says trump call with taiwan president wo...
2,fail! the trump organization’s credit score w...
3,zimbabwe military chief's china trip was norma...
4,the most uncourageous president ever receives ...
...,...
24348,mexico senate committee ok's air transport dea...
24349,breaking: hillary clinton’s state department g...
24350,trump breaks from stump speech to admire beaut...
24351,nfl player delivers courageous message: stop b...


##Remove URLs

In [30]:
def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', text)

In [31]:
train_df["url_removed"] = train_df["html_removed"].apply(remove_urls)
train_df["url_removed"]

,url_removed
0,palestinians switch off christmas lights in be...
1,china says trump call with taiwan president wo...
2,fail! the trump organization’s credit score w...
3,zimbabwe military chief's china trip was norma...
4,the most uncourageous president ever receives ...
...,...
24348,mexico senate committee ok's air transport dea...
24349,breaking: hillary clinton’s state department g...
24350,trump breaks from stump speech to admire beaut...
24351,nfl player delivers courageous message: stop b...


##Removing Punctuations

In [32]:
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [33]:
train_df["punctuation_removed"] = train_df["url_removed"].apply(remove_punctuation)
train_df["punctuation_removed"]

,punctuation_removed
0,palestinians switch off christmas lights in be...
1,china says trump call with taiwan president wo...
2,fail the trump organization’s credit score wi...
3,zimbabwe military chiefs china trip was normal...
4,the most uncourageous president ever receives ...
...,...
24348,mexico senate committee oks air transport deal...
24349,breaking hillary clinton’s state department ga...
24350,trump breaks from stump speech to admire beaut...
24351,nfl player delivers courageous message stop bl...


##Removing Numbers

In [34]:
def remove_numbers(text):
    return re.sub(r'\d+', '', text)

In [35]:
train_df["numbers_removed"] = train_df["punctuation_removed"].apply(remove_numbers)
train_df["numbers_removed"]

,numbers_removed
0,palestinians switch off christmas lights in be...
1,china says trump call with taiwan president wo...
2,fail the trump organization’s credit score wi...
3,zimbabwe military chiefs china trip was normal...
4,the most uncourageous president ever receives ...
...,...
24348,mexico senate committee oks air transport deal...
24349,breaking hillary clinton’s state department ga...
24350,trump breaks from stump speech to admire beaut...
24351,nfl player delivers courageous message stop bl...


##Tokenization

In [36]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [37]:
def tokenization(text):
    return word_tokenize(text)

In [38]:
train_df["tokenized"] = train_df["numbers_removed"].apply(tokenization)
train_df["tokenized"]

,tokenized
0,"[palestinians, switch, off, christmas, lights,..."
1,"[china, says, trump, call, with, taiwan, presi..."
2,"[fail, the, trump, organization, ’, s, credit,..."
3,"[zimbabwe, military, chiefs, china, trip, was,..."
4,"[the, most, uncourageous, president, ever, rec..."
...,...
24348,"[mexico, senate, committee, oks, air, transpor..."
24349,"[breaking, hillary, clinton, ’, s, state, depa..."
24350,"[trump, breaks, from, stump, speech, to, admir..."
24351,"[nfl, player, delivers, courageous, message, s..."


##Remove Stopwords

In [39]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [40]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [word for word in tokens if word.lower() not in stop_words]

In [41]:
train_df["stopwords_removed"] = train_df["tokenized"].apply(remove_stopwords)
train_df["stopwords_removed"]

,stopwords_removed
0,"[palestinians, switch, christmas, lights, beth..."
1,"[china, says, trump, call, taiwan, president, ..."
2,"[fail, trump, organization, ’, credit, score, ..."
3,"[zimbabwe, military, chiefs, china, trip, norm..."
4,"[uncourageous, president, ever, receives, cour..."
...,...
24348,"[mexico, senate, committee, oks, air, transpor..."
24349,"[breaking, hillary, clinton, ’, state, departm..."
24350,"[trump, breaks, stump, speech, admire, beautif..."
24351,"[nfl, player, delivers, courageous, message, s..."


##Lemmatization

In [42]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [43]:
lemmatizer = WordNetLemmatizer()

def lemmatization(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

In [44]:
train_df["lemmatized"] = train_df["stopwords_removed"].apply(lemmatization)
train_df["lemmatized"]

,lemmatized
0,"[palestinian, switch, christmas, light, bethle..."
1,"[china, say, trump, call, taiwan, president, w..."
2,"[fail, trump, organization, ’, credit, score, ..."
3,"[zimbabwe, military, chief, china, trip, norma..."
4,"[uncourageous, president, ever, receives, cour..."
...,...
24348,"[mexico, senate, committee, ok, air, transport..."
24349,"[breaking, hillary, clinton, ’, state, departm..."
24350,"[trump, break, stump, speech, admire, beautifu..."
24351,"[nfl, player, delivers, courageous, message, s..."


##Joining Text(Tokens)
                     1. After all these preprocessing steps, the data is still a list of words.
                     2. So we have to convert the list back into a sentence.

In [45]:
def join_text(tokens):
    return " ".join(tokens)

In [46]:
train_df["clean_text"] = train_df["lemmatized"].apply(join_text)

##Final Overview of Preprocesing

In [47]:
train_df[[
    "content",
    "lowercase",
    "tokenized",
    "stopwords_removed",
    "lemmatized",
    "clean_text"
]].head()

,content,lowercase,tokenized,stopwords_removed,lemmatized,clean_text
0,Palestinians switch off Christmas lights in Be...,palestinians switch off christmas lights in be...,"[palestinians, switch, off, christmas, lights,...","[palestinians, switch, christmas, lights, beth...","[palestinian, switch, christmas, light, bethle...",palestinian switch christmas light bethlehem a...
1,China says Trump call with Taiwan president wo...,china says trump call with taiwan president wo...,"[china, says, trump, call, with, taiwan, presi...","[china, says, trump, call, taiwan, president, ...","[china, say, trump, call, taiwan, president, w...",china say trump call taiwan president wont cha...
2,FAIL! The Trump Organization’s Credit Score W...,fail! the trump organization’s credit score w...,"[fail, the, trump, organization, ’, s, credit,...","[fail, trump, organization, ’, credit, score, ...","[fail, trump, organization, ’, credit, score, ...",fail trump organization ’ credit score make la...
3,Zimbabwe military chief's China trip was norma...,zimbabwe military chief's china trip was norma...,"[zimbabwe, military, chiefs, china, trip, was,...","[zimbabwe, military, chiefs, china, trip, norm...","[zimbabwe, military, chief, china, trip, norma...",zimbabwe military chief china trip normal visi...
4,THE MOST UNCOURAGEOUS PRESIDENT EVER Receives ...,the most uncourageous president ever receives ...,"[the, most, uncourageous, president, ever, rec...","[uncourageous, president, ever, receives, cour...","[uncourageous, president, ever, receives, cour...",uncourageous president ever receives courage a...
